In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import ast
import math
import matplotlib.patches as patches

PATH_ACTIONS    = "../../Data/Processed/actions_filtered.csv"
PATH_KEYPOINTS  = "../../Data/Unprocessed/keypoints.csv"
PATH_CLIPS      = "../../Data/Videos/Clips/"

PATH_TEST       = "../../Data/test_out.csv"
TEST_CLIP       = "1/11_Left.mp4"

EDGES = [
    (0, 1), (1, 3),
    (3, 5), (1, 2),
    (0, 2), (2, 4),
    (4, 6),
    (5, 7), (7, 9),     # left arm
    (6, 8), (8, 10),    # right arm
    (5, 6),             # shoulders
    (11, 12),           # hips
    (5, 11), (6, 12),   # torso
    (11, 13), (13, 15), # left leg
    (12, 14), (14, 16)  # right leg
]

In [21]:
def combine_df(
    df_frames: pd.DataFrame,
    df_actions: pd.DataFrame,
):
    """
    Combine frame-level pose data with action intervals so that:
    - Each row is one frame
    - LEFT and RIGHT actions are aligned to the same frame
    - Only frames where at least one fencer has an action are kept
    """

    df = df_frames.copy()

    # Initialize action columns
    df["left_action"] = None
    df["right_action"] = None

    # Process each fencer independently
    for side in ["LEFT", "RIGHT"]:
        actions = df_actions[df_actions["fencer"] == side]

        for _, act in actions.iterrows():
            mask = (
                (df["frame_idx"] >= act["start_frame"]) &
                (df["frame_idx"] <= act["end_frame"])
            )

            df.loc[mask, f"{side.lower()}_action"] = act["action"]

    # Keep only frames where at least one fencer has an action
    df = df[
        df["left_action"].notna() |
        df["right_action"].notna()
    ].reset_index(drop=True)

    df.rename(columns={
        "frame_idx": "frame",
        "left_label": "pred_left_label", 
        "right_label": "pred_right_label",
        "left_action": "left_label",
        "right_action": "right_label",
    }, inplace=True)
    df = df[["frame", "roi", "left_label", "pred_left_label", "right_label", "pred_right_label", "left_keypoints", "right_keypoints"]]

    return df

In [22]:
def draw_file(df, video_file, cols=2):
    frame_indices = sorted(df['frame'].unique())
    n_frames = len(frame_indices)
    rows = math.ceil(n_frames / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 6 * rows))
    axes = axes.flatten()

    video_path = PATH_CLIPS + video_file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Cannot open video {video_file}")
        return

    # Column mapping (edit here if names change)
    label_cols = {
        "LEFT": ("left_label", "pred_left_label"),
        "RIGHT": ("right_label", "pred_right_label"),
    }

    for i, frame_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ax = axes[i]
        ax.imshow(frame_rgb)
        ax.axis("off")

        row = df[df['frame'] == frame_idx]
        if row.empty:
            continue

        # ---- Draw ROI ----
        roi = row['roi'].values[0] if 'roi' in row else None
        if roi is not None:
            x1, y1, x2, y2 = roi
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor="yellow", facecolor="none"
            )
            ax.add_patch(rect)

        # ---- Draw fencers ----
        for side, color, x_text in [
            ("LEFT", (1, 0, 0), 25),
            ("RIGHT", (0, 1, 0), 1250),
        ]:
            kp_col = f"{side.lower()}_keypoints"
            if kp_col in row:
                keypoints = row[kp_col].values[0]
                flag = keypoints and len(keypoints) > 0
                if flag:
                    plot_skeleton_ax(ax, keypoints, color=color)

            # ---- Labels ----
            gt_col, pred_col = label_cols[side]
            gt = row[gt_col].values[0]
            pred = row[pred_col].values[0] if flag else "Missing Keypoints"

            label_str = [
                f"Real: {gt}",
                f"Pred: {pred}",
            ]

            if label_str:
                ax.text(
                    x_text, 130,
                    f"{side}\n" + "\n".join(label_str),
                    color=color,
                    fontsize=10,
                    weight="bold",
                    bbox=dict(facecolor="black", alpha=0.5, pad=3),
                )

            ax.set_title(f"Frame {frame_idx}")

    # Hide unused subplots
    for j in range(n_frames, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()
    cap.release()

def plot_skeleton_ax(ax, keypoints, color=(0,1,0), alpha=0.8):
    keypoints = np.array(keypoints)
    for (i,j) in EDGES:
        if i >= len(keypoints) or j >= len(keypoints):
            continue
        x1, y1 = keypoints[i]
        x2, y2 = keypoints[j]
        ax.plot([x1,x2],[y1,y2], color=color, alpha=alpha, linewidth=2)
    ax.scatter(keypoints[:,0], keypoints[:,1], color=color, s=20)


In [23]:
def draw_action(df, action_id, cols=3):
    df = df[df["action_id"] == action_id]
    file = df["file"].unique()[0]

    frame_indices = sorted(df['frame'].unique())
    n_frames = len(frame_indices)
    rows = math.ceil(n_frames / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 6 * rows))
    axes = axes.flatten()

    video_path = PATH_CLIPS + file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Cannot open video {file}")
        return

    for i, frame_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ax = axes[i]
        ax.imshow(frame_rgb)
        ax.axis("off")

        row = df[df['frame'] == frame_idx]
        if row.empty:
            continue

        # ---- Draw ROI ----
        roi = row['roi'].values[0] if 'roi' in row else None
        if roi is not None:
            x1, y1, x2, y2 = roi
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor="yellow", facecolor="none"
            )
            ax.add_patch(rect)

        row = df[(df['frame'] == frame_idx)]
        fencer = row['fencer'].values[0] 
        keypoints = row['keypoints'].values[0] 
        if keypoints and not (isinstance(keypoints, float) and np.isnan(keypoints)): 
            plot_skeleton_ax(ax, keypoints, color=(1,0,0) if fencer == 'LEFT' else (0,1,0))

        ax.set_title(f"Frame {frame_idx}")

    # Hide unused subplots
    for j in range(n_frames, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()
    cap.release()

def plot_skeleton_ax(ax, keypoints, color=(0,1,0), alpha=0.8):
    keypoints = np.array(keypoints)
    for (i,j) in EDGES:
        if i >= len(keypoints) or j >= len(keypoints):
            continue
        x1, y1 = keypoints[i]
        x2, y2 = keypoints[j]
        ax.plot([x1,x2],[y1,y2], color=color, alpha=alpha, linewidth=2)
    ax.scatter(keypoints[:,0], keypoints[:,1], color=color, s=20)


In [8]:
df_test = pd.read_csv(PATH_TEST)
df_actions = pd.read_csv(PATH_ACTIONS)

true_actions = df_actions[df_actions["file"] == TEST_CLIP]

df_test["roi"] = df_test["roi"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df_test["left_keypoints"] = df_test["left_keypoints"].apply(
    lambda x: [tuple(p) for p in ast.literal_eval(x)] if isinstance(x, str) else x
)

df_test["right_keypoints"] = df_test["right_keypoints"].apply(
    lambda x: [tuple(p) for p in ast.literal_eval(x)] if isinstance(x, str) else x
)

df = combine_df(df_test, true_actions)
df.head()

,frame,roi,left_label,pred_left_label,right_label,pred_right_label,left_keypoints,right_keypoints
0,22,"[0, 592, 1920, 851]",None,DEFENSE_DISTANCE_PULL,DEFENSE_DISTANCE_PULL,ATTACK_BASIC,"[(818.3582153320312, 608.701416015625), (819.3...","[(1118.645751953125, 633.8726806640625), (1123..."
1,23,"[0, 592, 1920, 851]",None,DEFENSE_DISTANCE_PULL,DEFENSE_DISTANCE_PULL,DEFENSE_DISTANCE_PULL,"[(833.2344360351562, 607.6680297851562), (834....","[(1137.9443359375, 629.45654296875), (1140.998..."
2,24,"[0, 592, 1920, 851]",None,DEFENSE_DISTANCE_PULL,DEFENSE_DISTANCE_PULL,DEFENSE_DISTANCE_PULL,"[(844.7388916015625, 611.5134887695312), (847....","[(1149.7093505859375, 626.2012939453125), (115..."
3,25,"[0, 592, 1920, 851]",None,DEFENSE_DISTANCE_PULL,DEFENSE_DISTANCE_PULL,DEFENSE_DISTANCE_PULL,"[(860.7244873046875, 615.3948364257812), (863....","[(1158.69091796875, 622.8067626953125), (1163...."
4,26,"[0, 592, 1920, 851]",None,DEFENSE_DISTANCE_PULL,DEFENSE_DISTANCE_PULL,DEFENSE_DISTANCE_PULL,"[(878.3161010742188, 611.19384765625), (879.59...","[(1168.0814208984375, 616.3419189453125), (117..."


In [ ]:
draw_file(df, TEST_CLIP, cols=4)

In [24]:
df_keypoints = pd.read_csv(PATH_KEYPOINTS)
df_actions = pd.read_csv(PATH_ACTIONS)

df_keypoints["keypoints"] = df_keypoints["keypoints"].apply(
    lambda x: [tuple(p) for p in ast.literal_eval(x)] if isinstance(x, str) else x
)

df_keypoints["roi"] = df_keypoints["roi"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df_merged = df_keypoints.merge(df_actions, on=["file", "fencer"], how="left")
df_merged = df_merged[
    (df_merged["frame"] >= df_merged["start_frame"]) &
    (df_merged["frame"] <= df_merged["end_frame"])
]

df_merged = df_merged[["file", "fencer", "action_id", "action", "frame", "start_frame", "end_frame", "confidence", "roi", "keypoints"]].reset_index(drop=True)

df_merged.head()

,file,fencer,action_id,action,frame,start_frame,end_frame,confidence,roi,keypoints
0,1/10_Left.mp4,LEFT,0,OTHER_NO_ACTION,0,0,22,0.898973,"[0, 590, 1920, 836]","[(651.2871704101562, 605.3470458984375), (654...."
1,1/10_Left.mp4,LEFT,0,OTHER_NO_ACTION,1,0,22,0.885350,"[0, 590, 1920, 836]","[(652.6170654296875, 606.501953125), (655.7320..."
2,1/10_Left.mp4,LEFT,0,OTHER_NO_ACTION,2,0,22,0.867700,"[0, 590, 1920, 836]","[(654.3292236328125, 608.1226806640625), (657...."
3,1/10_Left.mp4,LEFT,0,OTHER_NO_ACTION,3,0,22,0.881394,"[0, 590, 1920, 836]","[(658.3043823242188, 607.803466796875), (661.1..."
4,1/10_Left.mp4,LEFT,0,OTHER_NO_ACTION,4,0,22,0.856014,"[0, 590, 1920, 836]","[(664.1746826171875, 605.3037719726562), (665...."


In [ ]:
sample = df_merged.sample(50)

for action_id in sample["action_id"].unique():
    file = df_merged[df_merged["action_id"] == action_id]["file"].unique()[0]

    print(f"Action {action_id} in file {file}")
    draw_action(df_merged, action_id=action_id, cols=4)